In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2024-07-13 02:13:25--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.01s   

2024-07-13 02:13:25 (96.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))
print()
print(text[:1000])

length of dataset in characters:  1115394

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hun

In [ ]:
# getting all the unique characters in the text (you should use something like huggingface tokenizer for more complex datasets!)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"vocab size: {vocab_size}")
print(''.join(chars))

vocab size: 65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [ ]:
# mapping characters to integers, and vice versa
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# a simple character level encode/decode
encode = lambda sentence: [stoi[char] for char in sentence]
decode = lambda indexes: [itos[index] for index in indexes]

print(encode("Hello"))
print(decode([20, 43, 50, 50, 53]))

[20, 43, 50, 50, 53]
['H', 'e', 'l', 'l', 'o']


In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:14])
print("".join(decode(data[:14].numpy())))

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10])
First Citizen:


In [ ]:
split_size = int(0.9*len(data))
train_data = data[:split_size]
val_data = data[split_size:]
train_data.size(), val_data.size()

(torch.Size([1003854]), torch.Size([111540]))

In [ ]:
# example of data in 1 batch
batch_size = 8
train_data[:batch_size]

tensor([18, 47, 56, 57, 58,  1, 15, 47])

In [ ]:
# what the model should be trying to do
x = train_data[:batch_size]
y = train_data[1:batch_size+1]
for t in range(batch_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
torch.manual_seed(42)
seq_len = 8 # what is the maximum context length for predictions?


def get_sequenced_data(data, seq_len):

    # Calculate the number of batches
    num_batches = int(len(data)/seq_len)

    # Trim data to be a multiple of batch_size * seq_len
    trimmed_data_len = num_batches * seq_len
    data = data[:trimmed_data_len]

    # Reshape data into (num_batches, seq_len)
    data = data.view(num_batches, seq_len)

    return data


sequence_train_data = get_sequenced_data(train_data, seq_len)
sequence_val_data = get_sequenced_data(val_data, seq_len)

print(sequence_train_data.size())
print(sequence_train_data[0])

torch.Size([125481, 8])
tensor([18, 47, 56, 57, 58,  1, 15, 47])


In [ ]:
from torch.utils.data import Dataset, DataLoader

batch_size = 4 # how many independent sequences will we process in parallel?


class TinyShakespeareDataset(torch.utils.data.Dataset):

    def __init__(self, data):
        super().__init__()
        self.data = data

    def __len__(self):
        # Return the number of sequences
        return len(self.data)

    def __getitem__(self, idx):
        # Return the sequence at the given index
        return self.data[idx]


train_dataset = TinyShakespeareDataset(sequence_train_data)
val_dataset = TinyShakespeareDataset(sequence_val_data)
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
batch = next(iter(train_dataloader))

# Length of inputs/labels will be seq_len - 1, as inputs exclude last token from a seq, targets exclude first token from a seq
inputs = batch[:, :-1]
targets = batch[:, 1:]

print("inputs")
print(inputs)
print(inputs.size())
print()

print("targets")
print(targets)
print(targets.size())
print()

for b in range(batch_size): # batch dimension
    for t in range(seq_len-1): # time dimension
        context = inputs[b, :t+1]
        target = targets[b,t]
        print(f"when input is {context.tolist()} the target: {target}")
    print()

inputs
tensor([[56, 58,  1, 47, 52,  1, 17],
        [46, 43, 39, 60, 43, 52,  6],
        [ 1, 57, 47, 56,  6,  1, 39],
        [43, 56, 63,  1, 54, 39, 52]])
torch.Size([4, 7])

targets
tensor([[58,  1, 47, 52,  1, 17, 59],
        [43, 39, 60, 43, 52,  6,  0],
        [57, 47, 56,  6,  1, 39, 52],
        [56, 63,  1, 54, 39, 52, 45]])
torch.Size([4, 7])

when input is [56] the target: 58
when input is [56, 58] the target: 1
when input is [56, 58, 1] the target: 47
when input is [56, 58, 1, 47] the target: 52
when input is [56, 58, 1, 47, 52] the target: 1
when input is [56, 58, 1, 47, 52, 1] the target: 17
when input is [56, 58, 1, 47, 52, 1, 17] the target: 59

when input is [46] the target: 43
when input is [46, 43] the target: 39
when input is [46, 43, 39] the target: 60
when input is [46, 43, 39, 60] the target: 43
when input is [46, 43, 39, 60, 43] the target: 52
when input is [46, 43, 39, 60, 43, 52] the target: 6
when input is [46, 43, 39, 60, 43, 52, 6] the target: 0

when 

In [ ]:
# our input to the transformer
print(batch)
print(batch.size()) # size: (batch_size, seq_len)

tensor([[56, 58,  1, 47, 52,  1, 17, 59],
        [46, 43, 39, 60, 43, 52,  6,  0],
        [ 1, 57, 47, 56,  6,  1, 39, 52],
        [43, 56, 63,  1, 54, 39, 52, 45]])
torch.Size([4, 8])


## Implementing Self Attention

In [71]:
torch.manual_seed(42)

# simple example
B,T,C = 4,8,32 # batch_size, time (or seq_len), feature dimension
x = torch.randn(B,T,C)

print(x)
x.size()

tensor([[[ 1.9269,  1.4873,  0.9007,  ...,  0.0418, -0.2516,  0.8599],
         [-1.3847, -0.8712, -0.2234,  ...,  1.8446, -1.1845,  1.3835],
         [ 1.4451,  0.8564,  2.2181,  ..., -0.8278,  1.3347,  0.4835],
         ...,
         [-1.9006,  0.2286,  0.0249,  ..., -0.5558,  0.7043,  0.7099],
         [ 1.7744, -0.9216,  0.9624,  ..., -0.5003,  1.0350,  1.6896],
         [-0.0045,  1.6668,  0.1539,  ...,  0.5655,  0.5058,  0.2225]],

        [[-0.6855,  0.5636, -1.5072,  ...,  1.1566,  0.2691, -0.0366],
         [ 0.9733, -1.0151, -0.5419,  ..., -0.0553,  1.2049, -0.9825],
         [ 0.4334, -0.7172,  1.0554,  ..., -0.6766, -0.5730, -0.3303],
         ...,
         [ 0.6839, -1.3246, -0.5161,  ...,  1.1895,  0.7607, -0.7463],
         [-1.3839,  0.4869, -1.0020,  ...,  1.9535,  2.0487, -1.0880],
         [ 1.6217,  0.8513, -0.4005,  ...,  0.4232, -0.3389,  0.5180]],

        [[-1.3638,  0.1930, -0.6103,  ...,  0.6110,  1.2208, -0.6076],
         [-1.7376, -0.1254, -1.3658,  ..., -0

torch.Size([4, 8, 32])

In [72]:
H = 16 # head_size
key_matrix = nn.Linear(C, H, bias=False)
query_matrix = nn.Linear(C, H, bias=False)
value_matrix = nn.Linear(C, H, bias=False)

key = key_matrix(x)
query = query_matrix(x)
value = value_matrix(x)

key.size(), query.size(), value.size() # size: (B,T,H)

(torch.Size([4, 8, 16]), torch.Size([4, 8, 16]), torch.Size([4, 8, 16]))

In [74]:
from math import sqrt

scaled_raw_attn_values = torch.matmul(query, key.transpose(-2, -1)) * 1/sqrt(head_size) # (B, T, H) @ (B, H, T) ---> (B, T, T)

mask_indexes = torch.tril(torch.ones(T,T))
masked_attn_values = scaled_raw_attn_values.masked_fill(mask_indexes == 0, float('-inf'))
final_attn_values = F.softmax(masked_attn_values, dim=-1)

out = torch.matmul(final_attn_values, value) # (B, T, T) @ (B, T, H) ---> (B, T, H)

In [75]:
print(scaled_raw_attn_values[0])
scaled_raw_attn_values.size()

tensor([[-0.3332, -1.1723, -1.0216, -0.0545, -1.0950,  0.2735,  0.1340, -0.8490],
        [-0.6597,  0.7869, -1.2725,  1.6851,  0.1159,  0.5450,  0.2356, -0.1962],
        [ 0.3630, -1.5219,  0.7821, -1.7215, -0.3494,  0.2884, -0.1021, -1.4271],
        [-0.1001,  0.8649, -0.0335,  1.0221, -0.1350, -0.3078,  0.1440, -0.3019],
        [ 0.0136, -1.6202, -1.9888, -0.3327, -1.2507, -0.8928, -2.2674,  3.0561],
        [-0.5833,  1.2025, -0.3281,  0.9147,  0.9809, -0.4859,  1.7589,  0.1650],
        [ 1.1351, -1.9940,  1.5545, -1.8037, -0.5062, -2.6109, -1.0739,  1.6430],
        [-1.2784, -0.4554, -1.4118,  0.6392, -0.5780,  1.9291,  1.6689,  0.1103]],
       grad_fn=<SelectBackward0>)


torch.Size([4, 8, 8])

In [76]:
print(masked_attn_values[0])
masked_attn_values.size()

tensor([[-0.0833,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.1649,  0.1967,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.0907, -0.3805,  0.1955,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.0250,  0.2162, -0.0084,  0.2555,    -inf,    -inf,    -inf,    -inf],
        [ 0.0034, -0.4050, -0.4972, -0.0832, -0.3127,    -inf,    -inf,    -inf],
        [-0.1458,  0.3006, -0.0820,  0.2287,  0.2452, -0.1215,    -inf,    -inf],
        [ 0.2838, -0.4985,  0.3886, -0.4509, -0.1266, -0.6527, -0.2685,    -inf],
        [-0.3196, -0.1138, -0.3529,  0.1598, -0.1445,  0.4823,  0.4172,  0.0276]],
       grad_fn=<SelectBackward0>)


torch.Size([4, 8, 8])

In [77]:
print(final_attn_values[0])
final_attn_values.size()

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1905, 0.8095, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3742, 0.0568, 0.5690, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1288, 0.3380, 0.1376, 0.3956, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4311, 0.0841, 0.0582, 0.3049, 0.1217, 0.0000, 0.0000, 0.0000],
        [0.0537, 0.3205, 0.0694, 0.2404, 0.2568, 0.0592, 0.0000, 0.0000],
        [0.3396, 0.0149, 0.5165, 0.0180, 0.0658, 0.0080, 0.0373, 0.0000],
        [0.0165, 0.0375, 0.0144, 0.1120, 0.0332, 0.4069, 0.3136, 0.0660]],
       grad_fn=<SelectBackward0>)


torch.Size([4, 8, 8])

In [70]:
print(out[0])
print(out.size())

tensor([[ 0.7630, -0.2412, -0.4150,  0.3833,  0.5740, -1.6738,  0.7954,  0.6872,
         -0.3848,  0.5073, -0.5312, -0.1221,  0.0445,  1.2169,  0.9940,  1.5281],
        [ 0.4058, -0.0920, -0.7653, -0.5147,  0.1817, -0.4080,  0.0756, -0.7033,
         -0.0571,  0.3145,  0.3326,  0.0922,  0.1446,  0.5214,  0.3781, -0.1178],
        [ 0.2012,  0.0409, -0.1103,  0.3876,  0.6604, -0.8814,  0.2189,  0.0529,
         -0.4067,  0.3265, -0.1413, -0.2490, -0.4813,  0.5791,  0.9548,  1.0026],
        [ 0.0370,  0.2438, -0.1707, -0.0168, -0.0221, -0.3756, -0.1570, -0.6721,
         -0.1865,  0.2293,  0.1447,  0.1949,  0.2877,  0.4271,  0.1980,  0.0253],
        [ 0.2009,  0.1195, -0.2142,  0.3468,  0.1683, -0.8404,  0.0235, -0.0529,
         -0.1590,  0.2322, -0.2571,  0.0770,  0.1777,  0.6503,  0.4119,  0.5479],
        [-0.0446,  0.1640, -0.3607,  0.1286,  0.0677, -0.2968, -0.4088, -0.4496,
          0.0718,  0.0469,  0.0142,  0.1075,  0.0830,  0.2601,  0.1443, -0.1559],
        [ 0.1647,  0.0

Notes:



*   Attention doesn't have an idea of space, or sequence order
*   "self-attention" just means that the k and v has the same source as q.
*   In "cross-attention", q get produced from x, but k and v come from y, an external source (e.g. an encoder module), and so key_matrix and v_matrix can have another size according to y's shape
* "Scaled" attention additional divides attn_values by 1/sqrt(head_size). This makes it so when input q, k are unit variance, attn_values will be unit variance too and Softmax will stay diffuse and not saturate too much

# Implementing Our Final Model

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F



# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
dataset_seq_len = 33 # what is the maximum context length for predictions?
seq_len = dataset_seq_len - 1 # length of inputs/labels will be dataset_seq_len - 1, as inputs exclude last token from a seq, targets exclude first token from a seq
epochs = 25
eval_interval = 5
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
embedding_size = 128 # the vector size respresenting a single token
num_heads = 4
num_layers = 4
dropout = 0.1
torch.manual_seed(42)
torch.set_default_device(device)

In [3]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# getting all the unique characters in the text (you should use something like huggingface tokenizer for more complex datasets!)
chars = sorted(list(set(text)))
vocab_size = len(chars)

# mapping characters to integers, and vice versa
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# a simple character level encode/decode
encode = lambda sentence: [stoi[char] for char in sentence]
decode = lambda indexes: ''.join([itos[index] for index in indexes])

--2024-07-14 09:31:45--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.02s   

2024-07-14 09:31:45 (61.4 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [4]:
from torch.utils.data import Dataset, DataLoader


class TinyShakespeareDataset(Dataset):

    def __init__(self, data):
        super().__init__()
        self.data = data

    def __len__(self):
        # Return the number of sequences
        return len(self.data)

    def __getitem__(self, idx):
        # Return the sequence at the given index
        return self.data[idx]

    @staticmethod
    def get_sequenced_data(data, seq_len):

        # Calculate the number of batches
        num_batches = int(len(data)/seq_len)

        # Trim data to be a multiple of batch_size * seq_len
        trimmed_data_len = num_batches * seq_len
        data = data[:trimmed_data_len]

        # Reshape data into (num_batches, seq_len)
        data = data.view(num_batches, seq_len)

        return data


data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9*len(data)) # first 90% will be train, rest val

train_data = data[:split]
val_data = data[split:]

sequence_train_data = TinyShakespeareDataset.get_sequenced_data(train_data, dataset_seq_len)
sequence_val_data = TinyShakespeareDataset.get_sequenced_data(val_data, dataset_seq_len)

train_dataset = TinyShakespeareDataset(sequence_train_data)
val_dataset = TinyShakespeareDataset(sequence_val_data)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, generator=torch.Generator(device=device), shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, generator=torch.Generator(device=device), shuffle=True);

In [7]:
from math import sqrt


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, embedding_size, head_size, dropout):
        super().__init__()
        self.key = nn.Linear(embedding_size, head_size, bias=False)
        self.query = nn.Linear(embedding_size, head_size, bias=False)
        self.value = nn.Linear(embedding_size, head_size, bias=False)
        self.register_buffer("mask_indexes", torch.tril(torch.ones(seq_len, seq_len)))
        self.dropout = nn.Dropout(dropout)
        self.head_size = head_size

    def forward(self, x):

        B,T,C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        v = self.value(x) # (B,T,head_size)

        attn_values = torch.matmul(q, k.transpose(-2, -1)) * 1/sqrt(self.head_size)
        attn_values = attn_values.masked_fill(self.mask_indexes[:T, :T] == 0, float('-inf'))
        attn_values = F.softmax(attn_values, dim=-1)
        attn_values = self.dropout(attn_values)

        out = attn_values @ v # (B, T, T) @ (B, T, H) ---> (B, T, H)

        return out


class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, embedding_size, num_heads, head_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(embedding_size, head_size, dropout) for _ in range(num_heads)])
        self.projection = nn.Linear(embedding_size, embedding_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.projection(out) # what's this doing here?
        out = self.dropout(out)
        return out


class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, embedding_siz):
        super().__init__()
        self.ffd = nn.Sequential(
            nn.Linear(embedding_size, 4 * embedding_size),
            nn.ReLU(),
            nn.Linear(4 * embedding_size, embedding_size),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.ffd(x)


class TransformerBlock(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, embedding_size, num_heads, dropout):
        super().__init__()
        head_size = embedding_size // num_heads
        self.self_attention = MultiHeadAttention(embedding_size, num_heads, head_size, dropout)
        self.feed_forward = FeedFoward(embedding_size)
        self.layer_norm_1 = nn.LayerNorm(embedding_size)
        self.layer_norm_2 = nn.LayerNorm(embedding_size)

    def forward(self, x):
        # adding residule connections
        x = x + self.self_attention(self.layer_norm_1(x))
        x = x + self.feed_forward(self.layer_norm_2(x))
        return x


# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size, embedding_size, seq_len, num_heads, num_layers, dropout):
        super().__init__()

        self.token_embeddings = nn.Embedding(vocab_size, embedding_size)
        self.position_encoding = nn.Embedding(seq_len, embedding_size)
        self.transformer_blocks = nn.Sequential(*[TransformerBlock(embedding_size, num_heads, dropout) for _ in range(num_layers)])
        self.projection = nn.Linear(embedding_size, vocab_size)
        self.layer_norm_final = nn.LayerNorm(embedding_size)

    def forward(self, inputs):

        # inputs is (B,T)
        tok_emb = self.token_embeddings(inputs) # (B,T,C)
        pos_emb = self.position_encoding(torch.arange(inputs.size(1), device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.transformer_blocks(x) # (B,T,C)
        x = self.layer_norm_final(x) # (B,T,C)
        logits = self.projection(x) # (B,T,vocab_size)

        return logits

    def generate(self, input_tokens, seq_len, max_seq_len):

        # inputs is (B, T) array of indices in the current context
        # in a real GPT you'd also stop when the next token is a "[EOS]", but this is a toy example
        for _ in range(max_seq_len):

            # crop idx to the last seq_len tokens
            context_window_tokens = input_tokens[:, -seq_len:]

            # get the predictions
            logits = self(context_window_tokens)

            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)

            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # sample from the distribution, it's not auto pick the highest probability token!
            next_token_index = torch.multinomial(probs, num_samples=1) # (B, 1)

            # append sampled token index to the running sequence
            input_tokens = torch.cat((input_tokens, next_token_index), dim=1) # (B, T+1)

        return input_tokens

https://github.com/cezannec/capsule_net_pytorch/issues/4

In [8]:
# creating model
model = BigramLanguageModel(vocab_size, embedding_size, seq_len, num_heads, num_layers, dropout)
model = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# create a Pytorch loss function
loss_fn = torch.nn.CrossEntropyLoss()

# training loop
for epoch in range(epochs):
    train_loss = 0.0

    for batch in train_dataloader:

        inputs = batch[:, :-1]
        targets = batch[:, 1:]

        logits = model(inputs)

        B,T,C = logits.shape
        loss = loss_fn(logits.reshape(B*T,C), targets.reshape(B*T))

        # evaluate the loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # every once in a while evaluate the loss on val dataset
    if epoch % eval_interval == 0:
        model.eval()
        val_loss = 0.0

        with torch.no_grad():

            for batch in val_dataloader:

                inputs = batch[:, :-1]
                targets = batch[:, 1:]

                logits = model(inputs)

                B,T,C = logits.shape
                loss = loss_fn(logits.reshape(B*T,C), targets.reshape(B*T))
                val_loss += loss.item()

        model.train()

        print(f"step {epoch}: train loss {train_loss/len(train_dataloader):.4f}, val loss {val_loss/len(val_dataloader):.4f}")


torch.save(model.state_dict(), f'model_state_dict_{epochs}.pth')

step 0: train loss 2.0520, val loss 1.8882
step 5: train loss 1.5444, val loss 1.6731
step 10: train loss 1.4753, val loss 1.6213
step 15: train loss 1.4402, val loss 1.6114
step 20: train loss 1.4166, val loss 1.6036


In [18]:
# load model
model = BigramLanguageModel(vocab_size, embedding_size, seq_len, num_heads, num_layers, dropout)
model.load_state_dict(torch.load('model_state_dict_25.pth'))
model = model.to(device)
model.eval();

In [22]:
# generate from the model
def generate_from_context(model, context_chars, seq_len, max_seq_len=2000):
    context_tokens = torch.tensor(encode(context_chars), dtype=torch.long, device=device).unsqueeze(0)
    return decode(model.generate(context_tokens, seq_len, max_seq_len)[0].tolist())

line = "I must, in truth, the cause be why. Thy smiles have faded, lost on high."
output = generate_from_context(model, line, seq_len)
print(output)

I must, in truth, the cause be why. Thy smiles have faded, lost on high.

MARCIUS:
Sweet is of death! Which same thy convey'd with wondam.
Might not make his ta'er heavour's spoil--
Upon me drawn with deceived, but I must do
figue him so: if we met
for quantable divine it. First, smeats, save thy trumpet grieve fear with buriety.

LUCIO:
This way wish your princes.

GRENCE:
Look, and, I take King Richmond fear of
Getter to be rough,--'
You shall take hence in me:
Who wet have feast cannot be none and and affect
And not finger to under his brother,
Tell him outs with their foul follows down,
I like a watch: she aims to murtain
you so to be we neven to usurp, defend Warwick,
And tell him this Edward, net, yaung!
No knock, confessed.

MERCUS:
Paccoves I am no hear, stand and mad boad,
Found dreams notch'd it by my more taste
To her robbears out much, but whose this nine'ers do beseech your knaves forget, the law;
Set it in a Glace of heavens whipes
And him husbands, I fixt no dog fine
On 